# StayGuard — exploratory analysis

**Question:** how much of the commission we book actually lands, what makes bookings cancel, and where
should the business act?

> **Synthetic data, portfolio project.** The data is simulated with known effects planted
> (`docs/DATA_GENERATION.md`), so this analysis can be checked against ground truth. All money is in
> GBP; outcome rates use intakes that have already moved in.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.precision", 3)
pd.set_option("display.width", 120)

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
con = duckdb.connect(str(ROOT / "data" / "stayguard.duckdb"), read_only=True)


def q(sql):
    return con.execute(sql).df()


def sql_file(name):
    return q((ROOT / "sql" / f"{name}.sql").read_text(encoding="utf-8"))


q("SELECT snapshot_date, seed FROM meta_build")

,snapshot_date,seed
0,2026-10-31,20261031


## 1. Three currencies: totals must be converted first
Bookings are priced in GBP, AUD and EUR. Adding local amounts together is a classic silent error, so
every total in StayGuard uses the `*_gbp` columns of `v_bookings_gbp`.

In [2]:
q("""SELECT country, currency, COUNT(*) AS bookings,
       ROUND(SUM(commission_local)) AS commission_local, ROUND(SUM(commission_gbp)) AS commission_gbp
FROM v_bookings_gbp GROUP BY ALL ORDER BY commission_gbp DESC""")

,country,currency,bookings,commission_local,commission_gbp
0,UK,GBP,11503,1.095e+07,1.095e+07
1,AU,AUD,5593,1.061e+07,5.410e+06
2,IE,EUR,1146,1.439e+06,1.223e+06


In [3]:
naive, right = q(
    "SELECT SUM(commission_local) AS a, SUM(commission_gbp) AS b FROM v_bookings_gbp"
).iloc[0]
print(
    f"Naive local-currency total £{naive:,.0f} vs converted £{right:,.0f}: off by {naive / right - 1:.1%}"
)

Naive local-currency total £23,000,548 vs converted £17,587,316: off by 30.8%


## 2. Where does the funnel leak?

In [4]:
f = sql_file("01_funnel_by_city_intake").query("is_closed_intake")
stages = f[["leads", "contacted", "shortlisted", "booked", "moved_in"]].sum()
pd.DataFrame(
    {
        "students": stages,
        "share_of_enquiries": stages / stages["leads"],
        "kept_from_previous_step": stages / stages.shift(1),
    }
)

,students,share_of_enquiries,kept_from_previous_step
leads,55382,1.000,NaN
contacted,52014,0.939,0.939
shortlisted,29813,0.538,0.573
booked,17112,0.309,0.574
moved_in,12165,0.220,0.711


## 3. Why and when do bookings cancel?

In [5]:
sql_file("04_cancellation_reasons_pareto")[
    ["cancel_reason", "cancellations", "share_pct", "running_share_pct", "in_vital_few"]
]

,cancel_reason,cancellations,share_pct,running_share_pct,in_vital_few
0,found_cheaper,1271,25.7,25.7,True
1,visa_refused,1054,21.3,47.0,True
2,changed_plans,814,16.5,63.5,True
3,duplicate_booking,724,14.6,78.1,True
4,no_place,527,10.7,88.7,True
5,cooling_off,288,5.8,94.6,False
6,other,269,5.4,100.0,False


In [6]:
curve = sql_file("03_cohort_cancellation_curve").query("is_closed_intake")
by_bucket = curve.groupby(["bucket_order", "bucket_label"])["cancellations_in_bucket"].sum()
(by_bucket / by_bucket.sum()).rename("share_of_cancellations").to_frame()

,,share_of_cancellations
bucket_order,bucket_label,
1,0-7 days,0.164
2,8-30 days,0.245
3,31-60 days,0.217
4,61-90 days,0.146
5,91-180 days,0.201
6,181+ days,0.027


## 4. Who cancels? Visa status, deposit and lead time
These are the planted effects. Section 4c is the important subtlety: the action queue works on bookings
that are still alive 72 days before move-in, and for that group the lead-time effect reverses.

In [7]:
v = sql_file("09_source_country_visa_risk")
v.query("not small_sample").sort_values("cancellation_pct", ascending=False).head(6)[
    ["source_country", "visa_status_at_booking", "bookings", "cancellation_pct", "visa_refusal_pct"]
]

,source_country,visa_status_at_booking,bookings,cancellation_pct,visa_refusal_pct
17,Pakistan,not_applied,263,55.1,33.5
10,Nigeria,not_applied,312,52.2,32.1
7,Nepal,not_applied,246,44.3,19.5
4,India,not_applied,1118,43.2,16.5
18,Pakistan,pending,368,41.6,21.2
20,Saudi Arabia,not_applied,127,40.9,7.9


In [8]:
q("""SELECT deposit_paid, COUNT(*) AS bookings, ROUND(AVG(is_cancelled::INT), 3) AS cancellation_rate
FROM v_bookings_gbp WHERE is_closed_intake GROUP BY 1""")

,deposit_paid,bookings,cancellation_rate
0,False,3494,0.389
1,True,13618,0.264


In [9]:
# 4c. Lead time: all bookings vs bookings still alive 72 days before move-in (ADR-005)
band = """CASE WHEN days_booked_before_intake < 90 THEN '1: under 90 days'
               WHEN days_booked_before_intake < 180 THEN '2: 90-179 days' ELSE '3: 180+ days' END"""
q(f"""WITH b AS (SELECT *, {band} AS lead_time,
                       CAST(intake_start_date - 72 AS TIMESTAMP) AS as_of FROM v_bookings_gbp WHERE is_closed_intake)
SELECT lead_time,
       ROUND(AVG(is_cancelled::INT), 3) AS cancel_rate_all_bookings,
       ROUND(AVG(COALESCE(is_cancelled AND cancelled_at >= as_of, FALSE)::INT)
             FILTER (WHERE booked_at < as_of AND NOT COALESCE(is_cancelled AND cancelled_at < as_of, FALSE)), 3)
             AS cancel_rate_if_alive_72_days_out
FROM b GROUP BY 1 ORDER BY 1""")

,lead_time,cancel_rate_all_bookings,cancel_rate_if_alive_72_days_out
0,1: under 90 days,0.207,0.192
1,2: 90-179 days,0.288,0.159
2,3: 180+ days,0.418,0.132


## 5. Is postgraduate demand softening?

In [10]:
pg = sql_file("11_yoy_pg_demand_by_city")
seg = pg.groupby(["segment", "academic_year"])["pg_bookings"].sum().unstack()
seg.assign(change_2025_26=seg["2025-26"] / seg["2024-25"] - 1)

academic_year,2024-25,2025-26,2026-27,change_2025_26
segment,,,,
AU/IE,1081,1100,1128,0.018
London,511,500,532,-0.022
UK non-London,1228,892,941,-0.274


## 6. Operations: response time and agent workload

In [11]:
sql_file("08_response_time_vs_conversion")

,quintile,min_minutes,max_minutes,leads,booked,conversion_pct,pct_over_sla
0,1,1,30,10297,3759,36.5,0.0
1,2,31,58,10230,3717,36.3,0.0
2,3,59,104,10174,3604,35.4,0.0
3,4,105,206,10134,3517,34.7,0.0
4,5,207,8336,10135,2149,21.2,83.0


In [12]:
sql_file("06_agent_leaderboard").sort_values("load_days", ascending=False).head(4)[
    ["agent_name", "team", "open_bookings", "max_daily_capacity", "load_days", "cancellation_pct"]
]

,agent_name,team,open_bookings,max_daily_capacity,load_days,cancellation_pct
28,Gillian Jordan,UK North,55,20,2.75,34.5
27,Jacqueline Grant,UK North,56,22,2.55,35.3
14,Christopher Roberts,ANZ,55,26,2.12,26.1
9,James Aguilar,ANZ,49,26,1.88,31.0


## 7. What will land from the open intake?

In [13]:
sql_file("07_revenue_at_risk_by_city").head(5)[
    [
        "city",
        "open_bookings",
        "booked_commission_gbp",
        "expected_net_commission_gbp",
        "revenue_at_risk_gbp",
        "share_of_total_risk_pct",
    ]
]

,city,open_bookings,booked_commission_gbp,expected_net_commission_gbp,revenue_at_risk_gbp,share_of_total_risk_pct
0,London,220,216943.29,174878.65,42064.64,29.8
1,Sydney,139,102841.32,81981.37,20859.94,14.8
2,Melbourne,124,91309.97,70697.80,20612.17,14.6
3,Dublin,67,52508.28,41925.51,10582.78,7.5
4,Manchester,74,44856.74,35756.33,9100.41,6.5


In [14]:
ev = q("SELECT * FROM model_evaluation WHERE dataset LIKE 'validation%'")
ev.pivot_table(index="model", columns="metric", values="value")[
    ["roc_auc", "pr_auc", "lift_top_decile", "saved_gbp"]
]

metric,roc_auc,pr_auc,lift_top_decile,saved_gbp
model,,,,
baseline_segment_rate,0.625,0.216,1.602,39584.115
gradient_boosting,0.675,0.315,2.564,49284.555
leaky_demo,0.875,0.715,5.242,78136.188
logistic_regression,0.695,0.347,2.724,52085.470


## Takeaways
Computed from the cells above, so the numbers always match this run.

In [15]:
open_ = sql_file("07_revenue_at_risk_by_city")
enc, rar = open_["expected_net_commission_gbp"].sum(), open_["revenue_at_risk_gbp"].sum()
top = open_.iloc[0]
pareto = sql_file("04_cancellation_reasons_pareto")
vital_rows = pareto.loc[pareto["in_vital_few"]]
vital, vital_share = vital_rows["cancel_reason"].tolist(), vital_rows["running_share_pct"].iloc[-1]
print(
    f"1. Of £{enc + rar:,.0f} open commission, £{enc:,.0f} is expected to land; {top.city} holds {top.share_of_total_risk_pct:.0f}% of the £{rar:,.0f} at risk."
)
print(
    f"2. {len(vital)} of 7 reasons ({', '.join(vital)}) explain {vital_share:.1f}% of cancellations."
)
print(
    f"3. Biggest funnel leak: {(1 - stages['shortlisted'] / stages['contacted']):.0%} of contacted students never shortlist."
)
print(
    f"4. UK non-London PG bookings changed {seg.loc['UK non-London', '2025-26'] / seg.loc['UK non-London', '2024-25'] - 1:+.0%} in 2025-26."
)

1. Of £703,019 open commission, £561,969 is expected to land; London holds 30% of the £141,049 at risk.
2. 5 of 7 reasons (found_cheaper, visa_refused, changed_plans, duplicate_booking, no_place) explain 88.7% of cancellations.
3. Biggest funnel leak: 43% of contacted students never shortlist.
4. UK non-London PG bookings changed -27% in 2025-26.
